### ✍ Data Preparation

Encoding:
- Gender -> ohe
- Contract -> ohe

Feature (X):

[Age, Gender, Tenure, MonthlyCharges, Contract]

Target (Y):

[Churn: No=0, Yes=1]

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("ggplot")

# Dataset
telco_df = pd.read_csv(r"D:\Python Course\Telco Churn Risk Analytics\data\raw\synthetic_customer_churn_100k.csv")

#### Data Encoding & Preprocessing

In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
ohe = OneHotEncoder
ss = StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ohe(handle_unknown="ignore"), ["Gender", "Contract"]),
        ("num", ss(),["Age", "Tenure", "MonthlyCharges"])
    ]
)

#### Data Splitting

In [23]:
X = telco_df[["Age", "Gender", "Tenure", "MonthlyCharges", "Contract"]]
y = telco_df["Churn"].map({
    "No": 0,
    "Yes": 1
})

# X.head(1), y.head(1)
# X.shape, y.shape

#### Train Test Split

In [24]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y)

# X_train.shape, X_test.shape
# y_train.shape, y_test.shape

#### Pipeline Model_1

In [34]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Model_1 (Baseline)
model_1 = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000
    ))
])

# Model_1 Fit
model_1.fit(X_train, y_train)

# Model_1 Predict
yPred1 = model_1.predict(X_test)

print(f"Model_1 test result:\n{classification_report(y_test, yPred1, zero_division=True)}")

Model_1 test result:
              precision    recall  f1-score   support

           0       0.76      0.86      0.81     16714
           1       0.61      0.44      0.51      8286

    accuracy                           0.72     25000
   macro avg       0.68      0.65      0.66     25000
weighted avg       0.71      0.72      0.71     25000



#### Model 1 Result

Dari hasil train dan test model 1, didapatkan accuracy sekitar 0.72 accuracy dengan rata" berat F1-score 0.71. Lalu pada class 0 mendatkan performa yang lebih baik. F1-score mencapai 0.81, dibanding class 1 yang mendapat 0.51. Nilai recall rendah pada class 1 (0.44) menunjukkan kalau model 1 ini masih kesusahan untuk mengidentifikasi secara benar dari class 1.  

#### Pipeline Model_2

In [35]:
model_2 = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000
    ))
])

# Model_2 fit
model_2.fit(X_train, y_train)

# Model_2 predict
yPred2 = model_2.predict(X_test)

print(f"Model_2 test result:\n{classification_report(y_test, yPred2, zero_division=True)}")

Model_2 test result:
              precision    recall  f1-score   support

           0       0.83      0.68      0.74     16714
           1       0.52      0.71      0.60      8286

    accuracy                           0.69     25000
   macro avg       0.67      0.69      0.67     25000
weighted avg       0.73      0.69      0.70     25000



#### Model 2 Evaluation

Model 2 menggunakan `class_weight="balanced"` dibandingkan Model 1, accuracy mengalami penurunan dari 72% menjadi 69%. Namun, kemampuan model dalam mendeteksi customer yang melakukan churn meningkat secara signifikan, ditunjukkan oleh kenaikan recall class 1 dari 44% menjadi 71%.

Selain itu, F1-score untuk class 1 meningkat dari 0.51 menjadi 0.60. Meskipun precision mengalami penurunan dari 61% menjadi 52%, Model 2 lebih mampu mengidentifikasi customer yang berpotensi churn. Oleh karena itu, untuk tujuan churn detection, Model 2 menunjukkan performa yang lebih sesuai dibandingkan Model 1.